In [39]:
import json, os, glob, collections
from kaggle_secrets import UserSecretsClient
from huggingface_hub import hf_hub_download

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
REPO = "Exploration-Lab/CS-781-Capstone"
files = ["Train/fold1-text.json", "Train/fold1-gold.json",
         "Train/fold2-text.json", "Train/fold2-gold.json",
         "Train/labels.txt", "Test/fold3-text.json"]
P = {}
for f in files:
    P[f] = hf_hub_download(REPO, f"Phase_2_LNER/{f}", repo_type="dataset", token=HF_TOKEN)
    print(f, "->", os.path.getsize(P[f]) // 1024, "KB")

Train/fold1-text.json -> 1080 KB
Train/fold1-gold.json -> 449 KB
Train/fold2-text.json -> 1097 KB
Train/fold2-gold.json -> 464 KB
Train/labels.txt -> 0 KB
Test/fold3-text.json -> 1158 KB


In [2]:
t1 = json.load(open(P["Train/fold1-text.json"]))
g1 = json.load(open(P["Train/fold1-gold.json"]))
labels = open(P["Train/labels.txt"]).read().split()

print("labels:", labels)
print("text type:", type(t1).__name__, "| gold type:", type(g1).__name__)
k = list(t1)[0] if isinstance(t1, dict) else 0
print("first id:", k)
print("text sample:", repr(str(t1[k])[:400]))
print("gold sample:", json.dumps(g1[k] if isinstance(g1, dict) else g1[0], ensure_ascii=False)[:800])

labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']
text type: dict | gold type: dict
first id: 115651329
text sample: 'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal file'
gold sample: [[137, 153, "RESP", "STATE OF HARYANA"], [252, 261, "DATE", "19.8.2011"], [276, 308, "COURT", "High Court of Punjab and Haryana"], [359, 369, "COURT", "High Court"], [575, 585, "DATE", "18.11.1994"], [637, 644, "WIT", "Jagdish"], [695, 704, "WIT", "Mange Ram"], [792, 800, "APP", "Jage Ram"], [1122, 1130, "APP", "Jage Ram"], [1341, 1349, "APP", "Jage Ram"], [1373, 1380, "WIT", "Jagdish"], [1583, 15

In [3]:
def load(fold):
    t = json.load(open(P[f"Train/{fold}-text.json"]))
    g = json.load(open(P[f"Train/{fold}-gold.json"]))
    return t, g

for fold in ("fold1", "fold2"):
    t, g = load(fold)
    lens = [len(v) for v in t.values()]
    print(f"{fold}: {len(t)} docs | chars per doc: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

t3 = json.load(open(P["Test/fold3-text.json"]))
lens = [len(v) for v in t3.values()]
print(f"fold3 (test): {len(t3)} docs | chars: min {min(lens)}, median {sorted(lens)[len(lens)//2]}, max {max(lens)}")

fold1: 35 docs | chars per doc: min 10443, median 26600, max 106947
fold2: 35 docs | chars per doc: min 17216, median 26730, max 114044
fold3 (test): 35 docs | chars: min 0, median 28019, max 97103


In [4]:
def stats(fold):
    t, g = load(fold)
    cnt, lens, bad, edge_ws, overlaps = collections.Counter(), collections.defaultdict(list), 0, 0, 0
    for d, spans in g.items():
        text = t[d]
        spans = sorted(spans, key=lambda s: (s[0], s[1]))
        for i, (s, e, lab, ent) in enumerate(spans):
            cnt[lab] += 1
            lens[lab].append(e - s)
            if text[s:e] != ent: bad += 1                       # is `end` exclusive?
            if text[s:e] != text[s:e].strip(): edge_ws += 1      # spaces at span edges?
            if i and s < spans[i-1][1]: overlaps += 1            # overlapping / nested spans?
    print(f"\n{fold}: {sum(cnt.values())} entities | text mismatches {bad} | edge-whitespace {edge_ws} | overlaps {overlaps}")
    for lab in labels:
        L = lens[lab]
        print(f"  {lab:10s} {cnt[lab]:5d}   avg len {sum(L)/max(1,len(L)):6.1f}   max len {max(L) if L else 0}")
    return cnt

c1, c2 = stats("fold1"), stats("fold2")
print("\nempty test docs:", [d for d, v in t3.items() if not v.strip()])


fold1: 4120 entities | text mismatches 0 | edge-whitespace 0 | overlaps 0
  APP          207   avg len   12.8   max len 46
  RESP         191   avg len   14.7   max len 83
  A.COUNSEL     70   avg len   13.4   max len 25
  R.COUNSEL     87   avg len   11.5   max len 24
  JUDGE        123   avg len   14.0   max len 32
  WIT           67   avg len   10.7   max len 18
  AUTH         535   avg len   14.2   max len 89
  COURT        455   avg len   13.6   max len 59
  STAT         525   avg len   18.5   max len 107
  PREC         624   avg len   38.9   max len 195
  DATE         813   avg len   11.1   max len 28
  CASENO       423   avg len   20.2   max len 91

fold2: 4273 entities | text mismatches 0 | edge-whitespace 16 | overlaps 0
  APP          200   avg len   16.4   max len 35
  RESP         190   avg len   18.0   max len 40
  A.COUNSEL    125   avg len   10.8   max len 26
  R.COUNSEL     97   avg len   12.1   max len 29
  JUDGE        122   avg len   13.6   max len 32
  WIT         

In [5]:
import re
ROLE_LABELS = ("APP", "RESP", "WIT")

def name_consistency(fold):
    t, g = load(fold); tot = agree = 0; conflicts = collections.Counter()
    for d, spans in g.items():
        by = collections.defaultdict(collections.Counter)
        for s, e, l, ent in spans:
            if l in ROLE_LABELS + ("A.COUNSEL", "R.COUNSEL", "JUDGE"): by[ent][l] += 1
        for ent, c in by.items():
            n = sum(c.values()); tot += n; agree += c.most_common(1)[0][1]
            if len(c) > 1: conflicts[tuple(sorted(c))] += 1
    print(f"{fold}: person mentions {tot} | same role as the name's majority: {agree/tot:.3f} | conflicts {conflicts.most_common(4)}")

def unlabelled_repeats(fold, min_len=4):
    t, g = load(fold); lab = unl = 0
    for d, spans in g.items():
        text = t[d]; gold = {(s, e) for s, e, *_ in spans}
        for ent in {ent for s, e, l, ent in spans if l in ROLE_LABELS and len(ent) >= min_len}:
            for m in re.finditer(r"(?<!\w)" + re.escape(ent) + r"(?!\w)", text):
                if (m.start(), m.end()) in gold: lab += 1
                else: unl += 1
    print(f"{fold}: repeats of APP/RESP/WIT names → labelled {lab} | NOT labelled {unl} | precision ceiling {lab/(lab+unl):.3f}")

for f in ("fold1", "fold2"):
    name_consistency(f)
    unlabelled_repeats(f)

fold1: person mentions 745 | same role as the name's majority: 1.000 | conflicts []
fold1: repeats of APP/RESP/WIT names → labelled 460 | NOT labelled 98 | precision ceiling 0.824
fold2: person mentions 854 | same role as the name's majority: 1.000 | conflicts []
fold2: repeats of APP/RESP/WIT names → labelled 507 | NOT labelled 76 | precision ceiling 0.870


In [6]:
%%writefile a1_tokenizer.py
"""Expose your Assignment-1 BPE implementation to Assignment 3.

Reuse your A-1 tokenizer code/class here. The important change is that it must be
RETRAINED on the `train_text` supplied by Assignment 3 rather than reusing a tokenizer
trained on the Assignment-1 dataset.
"""
import re
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train(self,text,vocab_size):
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")
             
         

def train_tokenizer(train_text: str, vocab_size: int):
    """Train your A-1 BPE tokenizer on `train_text` and return the trained tokenizer.

    Required returned interface:
      tokenizer.encode(text) -> sequence of integer ids
      tokenizer.decode(ids) -> string
      tokenizer.vocab_size OR tokenizer.get_vocab_size()

    Replace this stub with your Assignment-1 tokenizer implementation/training call.
    """
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

Writing a1_tokenizer.py


In [7]:
import re
from a1_tokenizer import BPETokeniser
mp = glob.glob("/kaggle/input/**/legal_bpe_merges.json", recursive=True)[0]
raw = json.load(open(mp))
print(type(raw).__name__, len(raw))

list 65098


In [8]:
def load_tokenizer(raw):
    tok = BPETokeniser()
    if isinstance(raw, dict) and "merges" in raw:
        raw = raw["merges"]
    triples = []
    if isinstance(raw, dict):                                  # {"a,b": new_id}
        for k, v in raw.items():
            a, b = map(int, re.split(r"[,\s]+", k.strip("()[] ")))
            triples.append((a, b, int(v)))
    else:                                                      # list form
        for i, m in enumerate(raw):
            triples.append((int(m[0]), int(m[1]), int(m[2]) if len(m) == 3 else 256 + i))
    for a, b, new in sorted(triples, key=lambda t: t[2]):      # rebuild in learned order
        tok.merges[(a, b)] = new
        tok.vocab[new] = tok.vocab[a] + tok.vocab[b]
    return tok

tok = load_tokenizer(raw)
print("merges:", len(tok.merges), "| vocab:", tok.vocab_size)   # expect 65,098 | 65,354

merges: 65098 | vocab: 65354


In [9]:
from a1_tokenizer import PAT
import bisect

def encode_with_offsets(text):
    ids, offs = [], []
    for m in PAT.finditer(text):
        chunk, base = m.group(), m.start()
        b2c = []                                         # byte index -> char index within chunk
        for ci, ch in enumerate(chunk):
            b2c += [ci] * len(ch.encode("utf-8"))
        b2c.append(len(chunk))
        b = 0
        for t in tok.encode(chunk):
            nb = len(tok.vocab[t])
            ids.append(t)
            offs.append((base + b2c[b], base + b2c[min(b + nb, len(b2c) - 1)]))
            b += nb
    return ids, offs

# check: chunk-wise encoding must equal encoding the whole text
t1, g1 = load("fold1")
d0 = list(t1)[0]
ids, offs = encode_with_offsets(t1[d0][:5000])
print("same as tok.encode:", ids == tok.encode(t1[d0][:5000]), "|", len(ids), "tokens")
print("offsets cover the text:", offs[0][0] == 0, all(offs[i][1] == offs[i+1][0] for i in range(len(offs)-1)))
print([(t1[d0][s:e]) for s, e in offs[:15]])

same as tok.encode: True | 1053 tokens
offsets cover the text: True True
['REPORTABLE', ' IN', ' THE', ' SUPREME', ' COURT', ' OF', ' INDIA', ' CRIMINAL', ' APPELLATE', ' JURISDICTION', ' CRIMINAL', ' APPEAL', ' NO', '.', ' 92']


In [10]:
TAGS = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
T2I = {t: i for i, t in enumerate(TAGS)}

def core_start(text, s, e):                    # first non-space char of a token
    seg = text[s:e]
    return s + (len(seg) - len(seg.lstrip()))

def tag_doc(text, offs, spans):
    tags = [0] * len(offs)
    starts = [core_start(text, s, e) for s, e in offs]
    ends = [e for _, e in offs]
    for s, e, lab, *_ in spans:
        i = bisect.bisect_right(ends, s)       # first token ending after entity start
        first = True
        while i < len(offs) and starts[i] < e:
            if starts[i] < ends[i]:            # skip pure-whitespace tokens
                tags[i] = T2I[("B-" if first else "I-") + lab]; first = False
            i += 1
    return tags

def decode(text, offs, tags):
    spans, cur = [], None
    for i, t in enumerate(tags):
        name = TAGS[t]
        if name == "O":
            if cur: spans.append(cur); cur = None
            continue
        p, lab = name.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [core_start(text, *offs[i]), offs[i][1], lab]
        else:
            cur[1] = offs[i][1]
    if cur: spans.append(cur)
    out = []
    for s, e, lab in spans:                    # trim whitespace at the edges
        while e > s and text[e-1].isspace(): e -= 1
        if e > s: out.append([s, e, lab])
    return out

def strict_macro_f1(gold, pred):
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    for d in gold:
        G = {(s, e, l) for s, e, l, *_ in gold[d]}
        Pd = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for x in Pd: (tp if x in G else fp)[x[2]] += 1
        for x in G - Pd: fn[x[2]] += 1
    per = {}
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = 2 * p * r / (p + r) if p + r else 0.0
    return sum(per.values()) / len(labels), per

In [11]:
import time
ENC = {}
for fold in ("fold1", "fold2"):
    t, g = load(fold)
    t0 = time.time()
    pred = {}
    for d, text in t.items():
        ids, offs = encode_with_offsets(text)
        ENC[d] = (ids, offs)
        pred[d] = decode(text, offs, tag_doc(text, offs, g.get(d, [])))
    f, per = strict_macro_f1(g, pred)
    print(f"{fold}: oracle strict macro-F1 = {f:.4f}  ({time.time()-t0:.0f}s)")
    print("   ", {l: round(v, 3) for l, v in per.items()})

fold1: oracle strict macro-F1 = 0.9848  (1s)
    {'APP': 0.976, 'RESP': 1.0, 'A.COUNSEL': 1.0, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.981, 'COURT': 0.987, 'STAT': 0.989, 'PREC': 0.891, 'DATE': 1.0, 'CASENO': 0.994}
fold2: oracle strict macro-F1 = 0.9812  (1s)
    {'APP': 0.99, 'RESP': 0.963, 'A.COUNSEL': 0.98, 'R.COUNSEL': 1.0, 'JUDGE': 1.0, 'WIT': 1.0, 'AUTH': 0.991, 'COURT': 0.978, 'STAT': 0.974, 'PREC': 0.897, 'DATE': 1.0, 'CASENO': 1.0}


In [12]:
def boundary_misses(fold, max_show=12):
    t, g = load(fold)
    miss = collections.Counter(); examples = collections.defaultdict(list)
    for d, spans in g.items():
        text = t[d]; ids, offs = ENC[d]
        starts = {core_start(text, s, e) for s, e in offs}
        ends = {e for _, e in offs}
        for s, e, lab, ent in spans:
            ok_s, ok_e = s in starts, e in ends
            if ok_s and ok_e: continue
            kind = ("START" if not ok_s else "") + ("END" if not ok_e else "")
            miss[(lab, kind)] += 1
            if len(examples[lab]) < max_show:
                i = next(k for k, (a, b) in enumerate(offs) if b > (s if not ok_s else e - 1))
                examples[lab].append(f"{kind:8s} gold={text[s:e]!r}  context={text[max(0,s-15):e+15]!r}  "
                                     f"token={text[offs[i][0]:offs[i][1]]!r}")
    print(fold, "misaligned:", dict(miss))
    for lab, ex in examples.items():
        print(f"\n--- {lab}"); print("\n".join(ex))

boundary_misses("fold1")

fold1 misaligned: {('PREC', 'END'): 43, ('AUTH', 'END'): 10, ('STAT', 'END'): 6, ('COURT', 'END'): 6, ('APP', 'END'): 5, ('CASENO', 'END'): 1}

--- PREC
END      gold='State of M.P. vs. Kashiram & Ors.'  context='In the case of State of M.P. vs. Kashiram & Ors.[1], the scope '  token='.['
END      gold='Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]'  context='6 SC 1565] and Gambangi Appalaswamy Naldu and Others v. Behara Venkataramanayya Patro [AIR 1984- SC 1728]. Even otherwis'  token='].'
END      gold='Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]'  context='rlier cases in Shankarsan Dash Vs. Union of India [1991 (3) SCC 47]. Para 7 of the'  token='].'
END      gold='Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]'  context=' 4 SCC 268] or Jatendra Kumar vs. State of Punjab [(1985) 1 SCC 122]. Hence the con'  token='].'
END      gold='Hukam Singh v. State of Rajasthan AIR (1977 SC 1063)'  context='r person. (See Hukam Singh v

In [13]:
def build_known(folds):
    known = set()
    for fold in folds:
        _, g = load(fold)
        for spans in g.values():
            for s, e, lab, ent in spans:
                known.add((lab, ent))
    return known

def refine(text, s, e, lab, known):
    # 1) trailing ,;:([ glued after the entity
    while e > s and text[e-1] in ",;:([":
        e -= 1
    # 2) a '.' glued after a closing bracket:  ']. ' ')." 
    while e - s >= 2 and text[e-1] == "." and text[e-2] in ")]":
        e -= 1
    # 3) plural / adjective suffix, only if the shorter form is a known entity and the longer is not
    w = text[s:e]
    for suf in ("s", "al"):
        if w.endswith(suf) and (lab, w) not in known and (lab, w[:-len(suf)]) in known:
            e -= len(suf)
            break
    return s, e

def decode_refined(text, offs, tags, known):
    out = []
    for s, e, lab in decode(text, offs, tags):
        s, e = refine(text, s, e, lab, known)
        if e > s:
            out.append([s, e, lab])
    return out

KNOWN_ALL = build_known(["fold1", "fold2"])     # oracle only; in real CV build from the TRAINING fold only
for fold in ("fold1", "fold2"):
    t, g = load(fold)
    pred = {d: decode_refined(text, ENC[d][1], tag_doc(text, ENC[d][1], g.get(d, [])), KNOWN_ALL)
            for d, text in t.items()}
    f, per = strict_macro_f1(g, pred)
    print(f"{fold}: refined oracle = {f:.4f}  | PREC {per['PREC']:.3f}  AUTH {per['AUTH']:.3f}  "
          f"COURT {per['COURT']:.3f}  STAT {per['STAT']:.3f}")

fold1: refined oracle = 0.9929  | PREC 0.953  AUTH 0.998  COURT 0.989  STAT 1.000
fold2: refined oracle = 0.9900  | PREC 0.963  AUTH 1.000  COURT 0.980  STAT 0.985


In [14]:
# Header coverage: is the role information actually in the document's first H tokens?
ROLE4 = ("APP", "RESP", "A.COUNSEL", "R.COUNSEL")
def header_coverage(H_list=(128, 192, 256, 384, 512), W_first=1024):
    for H in H_list:
        names = hit = 0
        for fold in ("fold1", "fold2"):
            t, g = load(fold)
            for d, spans in g.items():
                ids, offs = ENC[d]
                cut = offs[min(H, len(offs)) - 1][1]                      # char end of token H
                for nm in {ent for s, e, l, ent in spans if l in ROLE4}:
                    names += 1; hit += nm in t[d][:cut]
        print(f"H={H:4d}: {hit}/{names} distinct party/counsel names ({hit/names:.1%}) appear in the header")
    beyond = total = 0
    for fold in ("fold1", "fold2"):
        t, g = load(fold)
        for d, spans in g.items():
            ends = [e for _, e in ENC[d][1]]
            for s, e, l, ent in spans:
                if l in ROLE4 + ("WIT", "JUDGE"):
                    total += 1; beyond += bisect.bisect_right(ends, s) >= W_first
    print(f"role mentions beyond the first {W_first} tokens (cannot see the header today): {beyond}/{total} ({beyond/total:.1%})")
header_coverage()

H= 128: 186/517 distinct party/counsel names (36.0%) appear in the header
H= 192: 228/517 distinct party/counsel names (44.1%) appear in the header
H= 256: 258/517 distinct party/counsel names (49.9%) appear in the header
H= 384: 280/517 distinct party/counsel names (54.2%) appear in the header
H= 512: 297/517 distinct party/counsel names (57.4%) appear in the header
role mentions beyond the first 1024 tokens (cannot see the header today): 893/1599 (55.8%)


In [15]:
import math, time, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, asdict
device = "cuda"
EOS_ID = 65354

In [16]:
class RMSNorm(nn.Module):
    def __init__(self,d_model,eps=1e-6):
        super().__init__()
        self.eps=eps
        self.g=nn.Parameter(torch.ones(d_model))
    def forward(self,x):
        meanSquare=x.pow(2).mean(dim=-1,keepdim=True)
        x=x*torch.rsqrt(meanSquare+self.eps)
        return x*self.g

In [17]:
def rope_tables(max_seq_len,d_head,base=10000.0):
    assert d_head%2==0
    i=torch.arange(0,d_head//2).float()
    theta=base**(-2*i/d_head)
    m=torch.arange(max_seq_len).float()
    angles=torch.outer(m,theta)
    return torch.cos(angles),torch.sin(angles)

In [18]:
def apply_rope(x,cos,sin):
    T=x.size(2)
    cos,sin=cos[:T],sin[:T]
    x1=x[...,0::2]
    x2=x[...,1::2]
    r1=x1*cos-x2*sin
    r2=x1*sin+x2*cos
    out=torch.stack([r1,r2],dim=-1)
    return out.flatten(-2)

In [19]:
def prefix_mask(T,prefix_lens,device=None):
    causal=torch.tril(torch.ones(T,T,dtype=torch.bool,device=device))
    pos=torch.arange(T,device=device)
    in_prefix=pos[None,:]<prefix_lens[:,None]
    both=in_prefix[:,:,None] & in_prefix[:,None,:]
    return (causal[None] | both)[:, None]
m = prefix_mask(6, torch.tensor([3, 0, 6]))
print(tuple(m.shape)); print(m[0, 0].int())
print(torch.equal(m[1, 0], torch.tril(torch.ones(6, 6, dtype=torch.bool))), bool(m[2, 0].all()), not bool(m[0, 0, 3, 4]))

(3, 1, 6, 6)
tensor([[1, 1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 0, 0, 0],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1]], dtype=torch.int32)
True True True


In [20]:
class GQAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        assert d_model%n_heads==0 and n_heads%n_kv_heads==0
        self.n_heads=n_heads
        self.n_kv_heads=n_kv_heads
        self.d_head=d_model // n_heads
        self.n_rep=n_heads // n_kv_heads
        self.scale=1/math.sqrt(self.d_head)
        self.q_proj=nn.Linear(d_model,d_model,bias=False)
        self.k_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.v_proj=nn.Linear(d_model,n_kv_heads*self.d_head,bias=False)
        self.out_proj=nn.Linear(d_model,d_model,bias=False)
        self.attn_dropout=nn.Dropout(dropout)
        self.resid_dropout=nn.Dropout(dropout)
        mask=torch.tril(torch.ones(max_seq_len,max_seq_len,dtype=torch.bool))
        self.register_buffer("causal_mask",mask,persistent=False)
        cos,sin=rope_tables(max_seq_len,self.d_head, rope_base)
        self.register_buffer("cos",cos,persistent=False)
        self.register_buffer("sin",sin,persistent=False)
    def forward(self,x,mask=None):
        B,T,C=x.shape
        q=self.q_proj(x).view(B,T,self.n_heads,self.d_head).transpose(1,2)
        k=self.k_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        v=self.v_proj(x).view(B,T,self.n_kv_heads,self.d_head).transpose(1,2)
        q=apply_rope(q,self.cos,self.sin)
        k=apply_rope(k,self.cos,self.sin)
        k=k.repeat_interleave(self.n_rep,dim=1)
        v=v.repeat_interleave(self.n_rep,dim=1)
        scores=(q*self.scale)@k.transpose(-2,-1)
        if mask is None:
            mask=self.causal_mask[:T,:T]
        scores=scores.masked_fill(~mask,float("-inf"))
        
        weights=self.attn_dropout(F.softmax(scores,dim=-1))
        out=(weights@v).transpose(1,2).contiguous().view(B,T,C)
        return self.resid_dropout(self.out_proj(out))

In [21]:
class SwiGLU(nn.Module):
    def __init__(self,d_model,d_ff,dropout=0.0):
        super().__init__()
        self.w1=nn.Linear(d_model,d_ff,bias=False)
        self.w3=nn.Linear(d_model,d_ff,bias=False)
        self.w2=nn.Linear(d_ff,d_model,bias=False)
        self.dropout=nn.Dropout(dropout)
    def forward(self,x):
        gate=F.silu(self.w1(x))
        return self.dropout(self.w2(gate*self.w3(x)))

In [22]:
from dataclasses import dataclass

@dataclass
class LlamaConfig:
    vocab_size:int
    d_model:int
    n_layers:int
    n_heads:int
    n_kv_heads:int
    d_ff:int
    max_seq_len:int=1024
    dropout:float=0.0
    rope_base:float=10000.0

class LlamaBlock(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.norm1=RMSNorm(cfg.d_model)
        self.attn=GQAttention(cfg.d_model,cfg.n_heads,cfg.n_kv_heads,cfg.max_seq_len,cfg.dropout,cfg.rope_base)
        self.norm2=RMSNorm(cfg.d_model)
        self.ffn=SwiGLU(cfg.d_model,cfg.d_ff,cfg.dropout)
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ffn(self.norm2(x))
        return x

class PrefixLM(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.cfg=cfg
        self.tok_emb=nn.Embedding(cfg.vocab_size,cfg.d_model)
        self.blocks=nn.ModuleList([LlamaBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f=RMSNorm(cfg.d_model)
        self.head=nn.Linear(cfg.d_model,cfg.vocab_size,bias=False)
        self.head.weight=self.tok_emb.weight
        nn.init.normal_(self.tok_emb.weight,mean=0.0,std=0.02)
    def forward(self,idx,prefix_lens=None):
        if prefix_lens is not None:
            mask=prefix_mask(idx.size(1),prefix_lens,idx.device)
        else:
            mask=None
        x=self.tok_emb(idx)
        for block in self.blocks: x=block(x,mask)
        x=self.norm_f(x)
        return self.head(x)

In [23]:
def load_encoder(path):
    ck = torch.load(path, map_location="cpu")
    cfg = LlamaConfig(**ck["cfg"])
    enc = PrefixLM(cfg); enc.load_state_dict(ck["model"])
    print(path.split("/")[-1], "| step", ck.get("step"), "| best_val", round(ck.get("best_val", float("nan")), 3),
          "| vocab", cfg.vocab_size, f"| {sum(p.numel() for p in enc.parameters()):,} params")
    del ck
    return enc, cfg

CK_A2 = glob.glob("/kaggle/input/**/arch2e2_ckpt.pt", recursive=True)[0]
CK_A3 = glob.glob("/kaggle/input/**/arch3_mlm_best.pt", recursive=True)[0]
for p in (CK_A2, CK_A3):
    e, c = load_encoder(p); del e

arch2e2_ckpt.pt | step 675 | best_val 2.991 | vocab 65355 | 184,909,824 params
arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params


In [24]:
W, STRIDE = 1024,512        # your choice (e.g. 512)

def make_windows(ids, tags=None):
    n, out, s = len(ids), [], 0
    while True:
        e = min(s + W, n)
        out.append((s, ids[s:e], None if tags is None else tags[s:e]))
        if e == n: break
        s += STRIDE
    return out

def build_split(folds):
    data = []
    for fold in folds:
        t, g = load(fold)
        for d, text in t.items():
            ids, offs = ENC[d]
            for s, wi, wt in make_windows(ids, tag_doc(text, offs, g.get(d, []))):
                data.append((d, s, wi, wt))
    return data

In [25]:
w1, w2 = build_split(["fold1"]), build_split(["fold2"])
print(f"windows: fold1 {len(w1)} | fold2 {len(w2)}")
print("tokens: fold1", sum(len(ENC[d][0]) for d in load('fold1')[0]),
      "| fold2", sum(len(ENC[d][0]) for d in load('fold2')[0]))
t, g = load("fold1"); ok = True
for d, text in t.items():
    ids, offs = ENC[d]; full = tag_doc(text, offs, g.get(d, []))
    cover = [False] * len(ids)
    for s, wi, wt in make_windows(ids, full):
        for k in range(len(wi)): cover[s + k] = True
        ok &= (wi == ids[s:s+len(wi)]) and (wt == full[s:s+len(wt)])
    ok &= all(cover)
print("full coverage + tags consistent:", ok)
cnt = collections.Counter(tg for *_, wt in w1 + w2 for tg in wt)
print({TAGS[k]: v for k, v in sorted(cnt.items())})

windows: fold1 423 | fold2 425
tokens: fold1 225233 | fold2 226960
full coverage + tags consistent: True
{'O': 778598, 'B-APP': 658, 'I-APP': 1218, 'B-RESP': 631, 'I-RESP': 1168, 'B-A.COUNSEL': 320, 'I-A.COUNSEL': 692, 'B-R.COUNSEL': 305, 'I-R.COUNSEL': 724, 'B-JUDGE': 318, 'I-JUDGE': 840, 'B-WIT': 340, 'I-WIT': 627, 'B-AUTH': 1884, 'I-AUTH': 2420, 'B-COURT': 1856, 'I-COURT': 2547, 'B-STAT': 2380, 'I-STAT': 5845, 'B-PREC': 2329, 'I-PREC': 20997, 'B-DATE': 2740, 'I-DATE': 10763, 'B-CASENO': 1233, 'I-CASENO': 9096}


In [26]:
class Tagger(nn.Module):
    def __init__(self,encoder,d_model,n_tags,dropout):
        super().__init__()
        self.encoder=encoder
        self.drop=nn.Dropout(dropout)
        self.cls=nn.Linear(d_model,n_tags)

    def forward(self,idx,prefix_lens=None):
        e=self.encoder
        mask=prefix_mask(idx.size(1),prefix_lens,idx.device) if prefix_lens is not None else None
        x=e.tok_emb(idx)
        for block in e.blocks:
            x = block(x, mask)
        x=e.norm_f(x)
        return self.cls(self.drop(x))

In [27]:
enc, ecfg = load_encoder(CK_A2)
tg = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device).eval()
x = torch.tensor([w1[0][2][:64]], device=device)
with torch.no_grad():
    out_causal = tg(x)                                                  # arch-2 style
    out_bidir  = tg(x, torch.tensor([x.size(1)], device=device))        # full bidirectional
print("shape:", tuple(out_causal.shape), "| expected (1, 64,", len(TAGS), ")")
print("head is new (not the LM head):", tg.cls.weight.shape == (len(TAGS), ecfg.d_model))
print("bidir differs from causal:", not torch.allclose(out_causal, out_bidir))
del tg, enc; torch.cuda.empty_cache()

arch2e2_ckpt.pt | step 675 | best_val 2.991 | vocab 65355 | 184,909,824 params
shape: (1, 64, 25) | expected (1, 64, 25 )
head is new (not the LM head): True
bidir differs from causal: True


In [28]:
def collate(batch):
    L = max(len(b[2]) for b in batch)
    x = torch.full((len(batch), L), EOS_ID, dtype=torch.long)
    y = torch.full((len(batch), L), -100, dtype=torch.long)
    pl = torch.tensor([len(b[2]) for b in batch])
    for i, (_, _, ids, tg) in enumerate(batch):
        x[i, :len(ids)] = torch.tensor(ids)
        if tg is not None: y[i, :len(tg)] = torch.tensor(tg)
    return x.to(device), y.to(device), pl.to(device)

def class_weights(windows, power):
    cnt = torch.zeros(len(TAGS))
    for *_, tg in windows:
        cnt += torch.bincount(torch.tensor(tg), minlength=len(TAGS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

@torch.no_grad()
def predict_logits(model, ids, bidir, B=8):
    wins = make_windows(ids)
    n = len(ids); out = torch.zeros(n, len(TAGS)); best = torch.full((n,), float("inf"))
    model.eval()
    for i in range(0, len(wins), B):
        chunk = wins[i:i+B]
        x, _, pl = collate([(None, s, w, None) for s, w, _ in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x, pl if bidir else None).float().cpu()
        for j, (s, w, _) in enumerate(chunk):
            L = len(w); pos = torch.arange(L)
            dist = (pos - (L - 1) / 2).abs()
            idx = s + pos; better = dist < best[idx]
            out[idx[better]] = lg[j, :L][better]; best[idx[better]] = dist[better]
    model.train()
    return out

def predict_texts(model, texts, bidir, known):
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        tags = predict_logits(model, ids, bidir).argmax(-1).tolist()
        pred[d] = decode_refined(text, offs, tags, known)
    return pred

def train_tagger(model, train_w, dev_fold, bidir, known, *, EPOCHS, LR_ENC, LR_HEAD,
                 WARMUP_FRAC, MICRO_B, CLASS_W_POWER, WD, CLIP=1.0, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    train_w = list(train_w)
    cw = class_weights(train_w, CLASS_W_POWER)
    head = list(model.cls.parameters()); hid = {id(p) for p in head}
    body = [p for p in model.parameters() if id(p) not in hid and p.requires_grad]
    opt = torch.optim.AdamW([{"params": body, "lr": LR_ENC}, {"params": head, "lr": LR_HEAD}], weight_decay=WD)
    per_ep = math.ceil(len(train_w) / MICRO_B); total = EPOCHS * per_ep; warm = max(1, int(WARMUP_FRAC * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda")
    dev_t, dev_g = load(dev_fold)
    best_f, best_state, best_per = -1.0, None, None
    model.train()
    for ep in range(EPOCHS):
        random.shuffle(train_w); t0 = time.time(); tot = 0.0
        for i in range(0, len(train_w), MICRO_B):
            x, y, pl = collate(train_w[i:i + MICRO_B])
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(x, pl if bidir else None)
            loss = F.cross_entropy(lg.float().view(-1, len(TAGS)), y.view(-1), weight=cw, ignore_index=-100)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            tot += loss.item()
        f, per = strict_macro_f1(dev_g, predict_texts(model, dev_t, bidir, known))
        print(f"epoch {ep+1}/{EPOCHS} | train loss {tot/per_ep:.4f} | dev strict macro-F1 {f:.4f} | {time.time()-t0:.0f}s")
        if f > best_f:
            best_f, best_per = f, per
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    print("BEST dev F1", round(best_f, 4), {l: round(v, 3) for l, v in best_per.items()})
    return best_f

In [29]:
# ===== fold-1 tagger for the submission-1 "before" analysis -> tagger_arch3mlm_fold1.pt =====
# Uses the ORIGINAL windows (this cell is before the header-windows cell on purpose).
# If the uploaded tagger_arch3mlm_fold1.pt is attached as an input, training is skipped.
_existing = glob.glob("/kaggle/**/tagger_arch3mlm_fold1.pt", recursive=True)
if _existing:
    print("found, skipping retrain:", _existing[0])
else:
    enc, ecfg = load_encoder(CK_A3)
    model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device)
    train_tagger(model, w1, "fold2", bidir=True, known=build_known(["fold1"]),
                 EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1,
                 MICRO_B=4, CLASS_W_POWER=0.2, WD=0.01)          # submission-1 settings
    torch.save(model.state_dict(), "/kaggle/working/tagger_arch3mlm_fold1.pt")
    print("saved /kaggle/working/tagger_arch3mlm_fold1.pt")
    del model, enc; torch.cuda.empty_cache()


found, skipping retrain: /kaggle/input/datasets/ashwaryrathoreh/njshvhcfghdhaj/tagger_arch3mlm_fold1.pt


# ===== EXPERIMENT 1: arch 2e2 (causal) =====
DROPOUT, EPOCHS, LR_ENC, LR_HEAD, WARMUP_FRAC, CLASS_W_POWER, WD = 0.1, 10, 1e-5, 1e-4, 0.1, 0.2, 0.01  # ← your values

enc, ecfg = load_encoder(CK_A2)
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=DROPOUT).to(device)
f2 = train_tagger(model, w1, "fold2", bidir=False, known=build_known(["fold1"]),
                  EPOCHS=EPOCHS, LR_ENC=LR_ENC, LR_HEAD=LR_HEAD, WARMUP_FRAC=WARMUP_FRAC,
                  MICRO_B=4, CLASS_W_POWER=CLASS_W_POWER, WD=WD)

torch.save(model.state_dict(), "/kaggle/working/tagger_arch2e2_fold1.pt")

# ===== EXPERIMENT 2: arch 3-MLM (bidirectional) — SAME values as Cell A =====
del model, enc; torch.cuda.empty_cache()           # free the GPU from experiment 1

enc, ecfg = load_encoder(CK_A3)
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=DROPOUT).to(device)
f3 = train_tagger(model, w1, "fold2", bidir=True, known=build_known(["fold1"]),
                  EPOCHS=EPOCHS, LR_ENC=LR_ENC, LR_HEAD=LR_HEAD, WARMUP_FRAC=WARMUP_FRAC,
                  MICRO_B=4, CLASS_W_POWER=CLASS_W_POWER, WD=WD)

print(f"\nRESULT  arch2e2 (causal): {f2:.4f}   |   arch3-MLM (bidirectional): {f3:.4f}")

import zipfile
torch.save(model.state_dict(), "/kaggle/working/tagger_arch3mlm_fold1.pt")

t3 = json.load(open(P["Test/fold3-text.json"]))
pred3 = predict_texts(model, t3, bidir=True, known=build_known(["fold1", "fold2"]))
json.dump(pred3, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("test docs:", len(pred3), "| entities predicted:", sum(len(v) for v in pred3.values()),
      "| empty doc 1305957 ->", pred3.get("1305957"))
      

In [30]:
# ---------- name propagation ----------
def propagate_names(text, spans, labels=ROLE_LABELS, min_len=4):
    votes = collections.defaultdict(collections.Counter)
    for s, e, l in spans:
        if l in labels: votes[text[s:e]][l] += 1
    role = {n: c.most_common(1)[0][0] for n, c in votes.items() if len(n) >= min_len}
    out = [list(x) for x in spans]
    for x in out:
        n = text[x[0]:x[1]]
        if x[2] in labels and n in role: x[2] = role[n]
    occ = [(s, e) for s, e, _ in out]
    for n, l in role.items():
        for m in re.finditer(r"(?<!\w)" + re.escape(n) + r"(?!\w)", text):
            s, e = m.start(), m.end()
            if not any(a < e and b > s for a, b in occ):
                out.append([s, e, l]); occ.append((s, e))
    return sorted(out)

# ---------- error breakdown ----------
def error_breakdown(gold, pred):
    stats = collections.defaultdict(collections.Counter)
    for d, G in gold.items():
        G = [(s, e, l) for s, e, l, *_ in G]; P = [tuple(p[:3]) for p in pred.get(d, [])]
        Gs, Ps = set(G), set(P)
        for s, e, l in G:
            if (s, e, l) in Ps: stats[l]["correct"] += 1; continue
            ov = [p for p in P if p[0] < e and p[1] > s]
            if not ov: stats[l]["missed"] += 1
            elif any(p[2] == l for p in ov): stats[l]["boundary"] += 1
            elif any((p[0], p[1]) == (s, e) for p in ov): stats[l]["wrong type"] += 1
            else: stats[l]["boundary+type"] += 1
        for p in P:
            if p not in Gs and not any(g[0] < p[1] and g[1] > p[0] for g in G):
                stats[p[2]]["spurious"] += 1
    cols = ["correct", "missed", "boundary", "wrong type", "boundary+type", "spurious"]
    print(f"{'label':10s}" + "".join(f"{c:>14s}" for c in cols))
    for l in labels:
        print(f"{l:10s}" + "".join(f"{stats[l][c]:>14d}" for c in cols))

# ---------- load the trained arch3-MLM tagger and test on fold 2 ----------
enc, ecfg = load_encoder(CK_A3)
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device)
model.load_state_dict(torch.load(glob.glob("/kaggle/**/tagger_arch3mlm_fold1.pt", recursive=True)[0], map_location="cpu"))

dev_t, dev_g = load("fold2")
pred = predict_texts(model, dev_t, True, build_known(["fold1"]))
f0, per0 = strict_macro_f1(dev_g, pred)
pred_p = {d: propagate_names(dev_t[d], p) for d, p in pred.items()}
f1, per1 = strict_macro_f1(dev_g, pred_p)

print(f"\nBEFORE {f0:.4f}  →  AFTER propagation {f1:.4f}\n")
for l in labels: print(f"  {l:10s} {per0[l]:.3f} → {per1[l]:.3f}")
print("\nERRORS AFTER PROPAGATION:")
error_breakdown(dev_g, pred_p)

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params

BEFORE 0.4148  →  AFTER propagation 0.4151

  APP        0.129 → 0.134
  RESP       0.090 → 0.086
  A.COUNSEL  0.207 → 0.207
  R.COUNSEL  0.234 → 0.234
  JUDGE      0.514 → 0.514
  WIT        0.230 → 0.234
  AUTH       0.332 → 0.332
  COURT      0.787 → 0.787
  STAT       0.736 → 0.736
  PREC       0.359 → 0.359
  DATE       0.926 → 0.926
  CASENO     0.432 → 0.432

ERRORS AFTER PROPAGATION:
label            correct        missed      boundary    wrong type boundary+type      spurious
APP                   42            30            46            52            30           191
RESP                  30            46            44            26            44           267
A.COUNSEL             29             0            31            63             2            21
R.COUNSEL             35             1            34            24             3            25
JUDGE                 85             0          

In [31]:
def show_errors(label, kind, n=12):
    shown = 0
    for d, G in dev_g.items():
        text = dev_t[d]; G = [(s, e, l) for s, e, l, *_ in G]; P = [tuple(p[:3]) for p in pred.get(d, [])]
        for p in P:
            if p[2] != label or p in set(G): continue
            ov = [g for g in G if g[0] < p[1] and g[1] > p[0]]
            if kind == "spurious" and not ov:
                print(f"SPURIOUS {label}: {text[p[0]:p[1]]!r}   …{text[max(0,p[0]-60):p[1]+40]!r}")
            elif kind == "boundary" and any(g[2] == label for g in ov):
                g = [g for g in ov if g[2] == label][0]
                print(f"BOUNDARY {label}: pred={text[p[0]:p[1]]!r}   gold={text[g[0]:g[1]]!r}")
            else: continue
            shown += 1
            if shown >= n: return

for lab in ("RESP", "APP"): show_errors(lab, "spurious"); print()
for lab in ("PREC", "CASENO"): show_errors(lab, "boundary"); print()
show_errors("PREC", "spurious")

SPURIOUS RESP: 'annum'   …'tual Matrix 3. The Complainant (Respondent No. 5 herein) Tarannum Akhtar @ Soni, was married to Md. Ikram'
SPURIOUS RESP: 'State of Bihar'   …'existing frame-work.” Contention made by Respondent No. 1 – State of Bihar 8. Respondent No. 1 herein i.e., the St'
SPURIOUS RESP: 'State of Bihar'   …'No. 1 – State of Bihar 8. Respondent No. 1 herein i.e., the State of Bihar, contends that the present FIR pertains'
SPURIOUS RESP: 'State of Bihar'   …'as dowry made in a previous FIR. Respondent No. 1 i.e., the State of Bihar, contends that the present FIR pertaine'
SPURIOUS RESP: 'of'   …'and in particular the field staff thereof. The Commissioner of Revenue Department was conferred with a'
SPURIOUS RESP: 'Secretary'   …'ing of such appointments. On the other hand, Md. A.R. Khan, Secretary (Revenue) has made many appointments of'
SPURIOUS RESP: 'Secretary'   …'ecent months against which I have been complaining that the Secretary (Revenue) has no power or authority to 

In [32]:
# ---------- constrained (Viterbi) decoding: valid BIO + whole-word consistency ----------
NT = len(TAGS)
ent_of = [None] + [TAGS[i][2:] for i in range(1, NT)]
NEG = -1e9
allow_start = np.ones((NT, NT), bool)          # token that STARTS a word
allow_cont  = np.zeros((NT, NT), bool)         # token that CONTINUES a word (e.g. "annum" after "Tar")
for p in range(NT):
    for c in range(NT):
        if TAGS[c].startswith("I-"):
            allow_start[p, c] = (p != 0 and ent_of[p] == ent_of[c])
        allow_cont[p, c] = (c == 0) if p == 0 else (TAGS[c] == "I-" + ent_of[p])
T_START = np.where(allow_start, 0.0, NEG); T_CONT = np.where(allow_cont, 0.0, NEG)
FIRST_OK = np.array([not t.startswith("I-") for t in TAGS])

def viterbi(logp, cont):
    n = logp.shape[0]
    score = np.where(FIRST_OK, logp[0], NEG)
    back = np.zeros((n, NT), np.int16)
    for i in range(1, n):
        cand = score[:, None] + (T_CONT if cont[i] else T_START)
        back[i] = cand.argmax(0)
        score = cand.max(0) + logp[i]
    tags = [int(score.argmax())]
    for i in range(n - 1, 0, -1):
        tags.append(int(back[i, tags[-1]]))
    return tags[::-1]

def fix_brackets(text, spans):                 # drop an unmatched closing ] or ) at the end
    out = []
    for s, e, l in spans:
        while e > s and text[e-1] in ")]" and text[s:e].count(text[e-1]) > text[s:e].count("(" if text[e-1] == ")" else "["):
            e -= 1
        if e > s: out.append([s, e, l])
    return out

def predict_texts_v(model, texts, bidir, known):
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = torch.log_softmax(predict_logits(model, ids, bidir), -1).numpy()
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp, cont), known))
    return pred

pred_v = predict_texts_v(model, dev_t, True, build_known(["fold1"]))
fv, perv = strict_macro_f1(dev_g, pred_v)
print(f"argmax {f0:.4f}  →  constrained decoding {fv:.4f}\n")
for l in labels: print(f"  {l:10s} {per0[l]:.3f} → {perv[l]:.3f}")
print("\nERRORS WITH CONSTRAINED DECODING:")
error_breakdown(dev_g, pred_v)

argmax 0.4148  →  constrained decoding 0.4914

  APP        0.129 → 0.214
  RESP       0.090 → 0.177
  A.COUNSEL  0.207 → 0.369
  R.COUNSEL  0.234 → 0.403
  JUDGE      0.514 → 0.557
  WIT        0.230 → 0.321
  AUTH       0.332 → 0.376
  COURT      0.787 → 0.808
  STAT       0.736 → 0.793
  PREC       0.359 → 0.417
  DATE       0.926 → 0.937
  CASENO     0.432 → 0.526

ERRORS WITH CONSTRAINED DECODING:
label            correct        missed      boundary    wrong type boundary+type      spurious
APP                   43            66            11            56            24            84
RESP                  35            69            13            42            31            80
A.COUNSEL             45             0             4            74             2            17
R.COUNSEL             50             1             2            42             2            17
JUDGE                 85             0            22            12             3            65
WIT                   30

In [33]:
import zipfile
# 1) quick check: does name propagation help now that predictions are clean?
pred_vp = {d: propagate_names(dev_t[d], p) for d, p in pred_v.items()}
print(f"dev: constrained {fv:.4f}  |  constrained + propagation {strict_macro_f1(dev_g, pred_vp)[0]:.4f}")

# 2) test-fold prediction with constrained decoding
t3 = json.load(open(P["Test/fold3-text.json"]))
pred3 = predict_texts_v(model, t3, True, build_known(["fold1", "fold2"]))
json.dump(pred3, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("test docs:", len(pred3), "| entities:", sum(len(v) for v in pred3.values()), "| empty doc:", pred3.get("1305957"))


dev: constrained 0.4914  |  constrained + propagation 0.4888
test docs: 35 | entities: 4671 | empty doc: []


In [1]:
# ---------- constrained (Viterbi) decoding: valid BIO + whole-word consistency ----------
NT = len(TAGS)
ent_of = [None] + [TAGS[i][2:] for i in range(1, NT)]
NEG = -1e9

NameError: name 'TAGS' is not defined

In [34]:
def show_wrong_type(label, n=12):
    k = 0
    for d, G in dev_g.items():
        text = dev_t[d]; P = [tuple(p[:3]) for p in pred_v.get(d, [])]
        for s, e, l, *_ in G:
            if l != label: continue
            for p in P:
                if (p[0], p[1]) == (s, e) and p[2] != l:
                    print(f"GOLD {l} → PRED {p[2]}: {text[s:e]!r}   …{text[max(0,s-80):e+30]!r}")
                    k += 1
            if k >= n: return
show_wrong_type("A.COUNSEL"); print(); show_wrong_type("APP"); print(); show_wrong_type("WIT")

GOLD A.COUNSEL → PRED R.COUNSEL: 'Aloke Ghosh'   …'ng dissatisfied with the annual valuation assessed by the learned Tribunal. Mr. Aloke Ghosh, learned Advocate appearing o'
GOLD A.COUNSEL → PRED R.COUNSEL: 'Ghosh'   …'A.A. 272 of 1996 is identical and comparable with the premises in question. Mr. Ghosh further contends that the ass'
GOLD A.COUNSEL → PRED R.COUNSEL: 'Ghosh'   …'f the property in question is fourth quarter of 2009-10. Thus, according to Mr. Ghosh, when there is a long gap of '
GOLD A.COUNSEL → PRED R.COUNSEL: 'Ghosh'   …'s the basis for assessing the annual valuation of the property in question. Mr. Ghosh also contends that the annual'
GOLD A.COUNSEL → PRED R.COUNSEL: 'Ghosh'   …'h the valuation fixed by the learned Tribunal in respect of other premises. Mr. Ghosh further contends that the lea'
GOLD A.COUNSEL → PRED R.COUNSEL: 'Ghosh'   …'ot be applied to arrive at a conclusion in respect of a different premises. Mr. Ghosh contends that the initial onu'
GOLD A.COUNSEL →

In [35]:
ROLE_SET = ("APP", "RESP", "A.COUNSEL", "R.COUNSEL")
A_W = r"(appellant|petitioner|applicant|plaintiff|complainant)"
R_W = r"(respondent|opposite part|defendant)"
LAB = {"C": {"A": "A.COUNSEL", "R": "R.COUNSEL"}, "P": {"A": "APP", "R": "RESP"}}

def side_cues(text, s, e):
    cues = []
    after = text[e:e+150].lower(); tail = text[max(0, s-40):s].lower()
    m_a = re.search(r"\b(for|behalf of)\b[^;]{0,60}?" + A_W, after)     # counsel: "...appearing for the appellant"
    m_r = re.search(r"\b(for|behalf of)\b[^;]{0,60}?" + R_W, after)
    if m_a and (not m_r or m_a.start() < m_r.start()): cues.append(("C", "A", 3))
    elif m_r: cues.append(("C", "R", 3))
    if re.search(A_W + r"[^;\n]{0,25}$", tail) or "instance of" in tail: cues.append(("P", "A", 3))   # "appellant No.4 - X"
    elif re.search(R_W + r"[^;\n]{0,25}$", tail): cues.append(("P", "R", 3))
    if s < 2500:                                                        # header "X Vs. Y"
        if re.match(r"\W{0,5}(&\s*ors\.?)?\W{0,3}(vs\.?|v\.|versus)\b", text[e:e+20].lower()): cues.append(("P", "A", 5))
        if re.search(r"\b(vs\.?|v\.|versus)\W{0,5}$", text[max(0, s-15):s].lower()): cues.append(("P", "R", 5))
    return cues

def apply_roles(text, spans):
    votes = collections.defaultdict(collections.Counter)
    for s, e, l in spans:
        if l not in ROLE_SET: continue
        k = text[s:e]; votes[k][l] += 1
        for g, side, w in side_cues(text, s, e):
            votes[k][LAB[g][side]] += w
    role = {k: c.most_common(1)[0][0] for k, c in votes.items()}
    sur = collections.defaultdict(set)
    for k, l in role.items():
        w = k.split()
        if len(w) >= 2 and len(w[-1]) >= 4: sur[w[-1]].add(l)
    sur = {k: next(iter(v)) for k, v in sur.items() if len(v) == 1}
    out = []
    for s, e, l in spans:
        if l in ROLE_SET:
            k = text[s:e]
            l = role.get(k, l)
            if len(k.split()) == 1 and k in sur: l = sur[k]
        out.append([s, e, l])
    return out

pred_r = {d: apply_roles(dev_t[d], p) for d, p in pred_v.items()}
fr, perr = strict_macro_f1(dev_g, pred_r)
print(f"constrained {fv:.4f}  →  + role rules {fr:.4f}\n")
for l in ROLE_SET: print(f"  {l:10s} {perv[l]:.3f} → {perr[l]:.3f}")

constrained 0.4914  →  + role rules 0.5053

  APP        0.214 → 0.272
  RESP       0.177 → 0.186
  A.COUNSEL  0.369 → 0.417
  R.COUNSEL  0.403 → 0.455


In [40]:
# ===== header-as-prefix windows (submission 2) =====
HDR = 192
# <- YOUR choice from Step 1 (e.g. the smallest H that covers most names)
assert HDR is not None
W_TOTAL = W         # encoder positions available (1024, same as before)

def make_windows_hdr(ids, tags=None, H=None):
    """Returns (start, x_ids, y_tags, P): x = [header + SEP] + slice for every window after the
    first; P = prefix length (0 for the first window, which already contains the header)."""
    H = HDR if H is None else H
    n = len(ids)
    e = min(W_TOTAL, n)
    out = [(0, ids[:e], None if tags is None else tags[:e], 0)]
    if e == n:
        return out
    hdr = ids[:H] + [EOS_ID]; P = len(hdr)
    WB = W_TOTAL - P; SB = WB // 2                      # body length and stride (50% overlap)
    s = SB
    while True:
        e = min(s + WB, n)
        out.append((s, hdr + ids[s:e], None if tags is None else [-100] * P + tags[s:e], P))
        if e == n:
            break
        s += SB
    return out

def build_split_hdr(folds):
    data = []
    for fold in folds:
        t, g = load(fold)
        for d, text in t.items():
            ids, offs = ENC[d]
            for s, x, y, P in make_windows_hdr(ids, tag_doc(text, offs, g.get(d, []))):
                data.append((d, s, x, y))
    return data

def class_weights(windows, power):                      # same as before, ignoring -100 labels
    cnt = torch.zeros(len(TAGS))
    for *_, tg in windows:
        t_ = torch.tensor(tg); t_ = t_[t_ >= 0]
        cnt += torch.bincount(t_, minlength=len(TAGS)).float()
    w = (cnt.sum() / cnt.clamp(min=1)) ** power
    return (w / w.mean()).to(device)

@torch.no_grad()
def predict_logits(model, ids, bidir, B=8):              # same centre-of-window stitching as before
    wins = make_windows_hdr(ids)
    n = len(ids); out = torch.zeros(n, len(TAGS)); best = torch.full((n,), float("inf"))
    model.eval()
    for i in range(0, len(wins), B):
        chunk = wins[i:i + B]
        x, _, pl = collate([(None, s, xw, None) for s, xw, _, _ in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x, pl if bidir else None).float().cpu()
        for j, (s, xw, _, P) in enumerate(chunk):
            L = len(xw) - P; pos = torch.arange(L)
            dist = (pos - (L - 1) / 2).abs()
            idx = s + pos; better = dist < best[idx]
            out[idx[better]] = lg[j, P:P + L][better]; best[idx[better]] = dist[better]
    model.train()
    return out

# checks: full coverage, labels aligned, nothing longer than the encoder allows
t, g = load("fold1"); ok = True
for d, text in t.items():
    ids, offs = ENC[d]; full = tag_doc(text, offs, g.get(d, [])); cover = [False] * len(ids)
    for s, x, y, pk in make_windows_hdr(ids, full):
        L = len(x) - pk
        ok &= len(x) <= W_TOTAL and x[pk:] == ids[s:s + L] and y[pk:] == full[s:s + L] and all(v == -100 for v in y[:pk])
        for k in range(L): cover[s + k] = True
    ok &= all(cover)
print("header windows OK:", ok)
w1h, w2h = build_split_hdr(["fold1"]), build_split_hdr(["fold2"])
print(f"windows: fold1 {len(w1h)} | fold2 {len(w2h)}   (old: {len(w1)} | {len(w2)})")

header windows OK: True
windows: fold1 525 | fold2 530   (old: 423 | 425)


In [44]:
# ===== EXPERIMENT 3: arch3-MLM + header-prefix windows (only change vs submission 1) =====
try: del model, enc
except NameError: pass
torch.cuda.empty_cache()
enc, ecfg = load_encoder(CK_A3)
assert ecfg.max_seq_len >= W_TOTAL
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device)
f_h = train_tagger(model, w1h, "fold2", bidir=True, known=build_known(["fold1"]),
                   EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1,
                   MICRO_B=4, CLASS_W_POWER=0.2, WD=0.01)          # your submission-1 settings
torch.save(model.state_dict(), "/kaggle/working/tagger_arch3mlm_hdr_fold1.pt")

dev_t, dev_g = load("fold2")
pred_v = predict_texts_v(model, dev_t, True, build_known(["fold1"]))
fv, perv = strict_macro_f1(dev_g, pred_v)
pred_r = {d: apply_roles(dev_t[d], p) for d, p in pred_v.items()}
fr, perr = strict_macro_f1(dev_g, pred_r)
print(f"\nDEV fold2 | constrained {fv:.4f} | + role rules {fr:.4f}   (submission-1 pipeline: 0.4914 | 0.5053)")
for l in labels: print(f"  {l:10s} {perv[l]:.3f} -> roles {perr[l]:.3f}")
error_breakdown(dev_g, pred_r)

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
epoch 1/10 | train loss 0.9465 | dev strict macro-F1 0.2678 | 102s
epoch 2/10 | train loss 0.1597 | dev strict macro-F1 0.3540 | 102s
epoch 3/10 | train loss 0.0969 | dev strict macro-F1 0.3905 | 102s
epoch 4/10 | train loss 0.0647 | dev strict macro-F1 0.3881 | 102s
epoch 5/10 | train loss 0.0482 | dev strict macro-F1 0.4067 | 102s
epoch 6/10 | train loss 0.0340 | dev strict macro-F1 0.4088 | 102s
epoch 7/10 | train loss 0.0241 | dev strict macro-F1 0.4098 | 102s
epoch 8/10 | train loss 0.0175 | dev strict macro-F1 0.4162 | 102s
epoch 9/10 | train loss 0.0146 | dev strict macro-F1 0.4143 | 102s
epoch 10/10 | train loss 0.0119 | dev strict macro-F1 0.4184 | 102s
BEST dev F1 0.4184 {'APP': 0.129, 'RESP': 0.087, 'A.COUNSEL': 0.156, 'R.COUNSEL': 0.283, 'JUDGE': 0.558, 'WIT': 0.197, 'AUTH': 0.342, 'COURT': 0.807, 'STAT': 0.724, 'PREC': 0.371, 'DATE': 0.942, 'CASENO': 0.423}

DEV fold2 | constrained 0.4876 | + 

In [41]:
# ===== FINAL submission 2: arch3-MLM + header windows, trained on BOTH folds =====
import zipfile
del model; torch.cuda.empty_cache()
enc, ecfg = load_encoder(CK_A3)
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device)
train_tagger(model, w1h + w2h, "fold2", bidir=True, known=build_known(["fold1", "fold2"]),
             EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1, MICRO_B=4, CLASS_W_POWER=0.2, WD=0.01)
torch.save(model.state_dict(), "/kaggle/working/tagger_arch3mlm_hdr_both.pt")   # (its "dev" F1 is on training data)
t3 = json.load(open(P["Test/fold3-text.json"]))
pred3 = predict_texts_v(model, t3, True, build_known(["fold1", "fold2"]))
pred3 = {d: apply_roles(t3[d], p) for d, p in pred3.items()}
json.dump(pred3, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("SUBMISSION 2 | test docs:", len(pred3), "| entities:", sum(len(v) for v in pred3.values()),
      "| empty doc:", pred3.get("1305957"))


arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
epoch 1/10 | train loss 0.7892 | dev strict macro-F1 0.4270 | 183s
epoch 2/10 | train loss 0.1413 | dev strict macro-F1 0.6118 | 182s
epoch 3/10 | train loss 0.0866 | dev strict macro-F1 0.7410 | 182s
epoch 4/10 | train loss 0.0592 | dev strict macro-F1 0.7806 | 182s
epoch 5/10 | train loss 0.0405 | dev strict macro-F1 0.8321 | 182s
epoch 6/10 | train loss 0.0281 | dev strict macro-F1 0.8855 | 182s
epoch 7/10 | train loss 0.0209 | dev strict macro-F1 0.9094 | 182s
epoch 8/10 | train loss 0.0154 | dev strict macro-F1 0.9362 | 182s
epoch 9/10 | train loss 0.0111 | dev strict macro-F1 0.9381 | 182s
epoch 10/10 | train loss 0.0094 | dev strict macro-F1 0.9431 | 182s
BEST dev F1 0.9431 {'APP': 0.948, 'RESP': 0.897, 'A.COUNSEL': 0.958, 'R.COUNSEL': 0.96, 'JUDGE': 0.917, 'WIT': 0.992, 'AUTH': 0.977, 'COURT': 0.953, 'STAT': 0.944, 'PREC': 0.915, 'DATE': 0.986, 'CASENO': 0.872}
SUBMISSION 2 | test docs: 35 | entiti

# ===== FINAL: arch3-MLM trained on BOTH folds → test prediction =====
import zipfile
del model; torch.cuda.empty_cache()
enc, ecfg = load_encoder(CK_A3)
model = Tagger(enc, ecfg.d_model, len(TAGS), dropout=0.1).to(device)
f_all = train_tagger(model, w1 + w2, "fold2", bidir=True, known=build_known(["fold1", "fold2"]),
                     EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1,
                     MICRO_B=4, CLASS_W_POWER=0.2, WD=0.01)          # your same settings
torch.save(model.state_dict(), "/kaggle/working/tagger_arch3mlm_both.pt")

t3 = json.load(open(P["Test/fold3-text.json"]))
pred3 = predict_texts_v(model, t3, True, build_known(["fold1", "fold2"]))
json.dump(pred3, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("FINAL test docs:", len(pred3), "| entities:", sum(len(v) for v in pred3.values()))


**(disabled — submission-1 test rewrite; kept for reference)**

```python
pred3r = {d: apply_roles(t3[d], p) for d, p in pred3.items()}
json.dump(pred3r, open("/kaggle/working/prediction.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission.zip", "w") as z:
    z.write("/kaggle/working/prediction.json", arcname="prediction.json")
print("test docs:", len(pred3r), "| entities:", sum(len(v) for v in pred3r.values()), "| empty doc:", pred3r.get("1305957"))
```

In [43]:
# ===== A. gazetteer post-pass: per-label dev check (no training) =====
def build_gazetteer(folds, min_count=2, min_len=4):
    """Entity strings that always carry ONE label in the given folds and occur >= min_count times."""
    lab_of = collections.defaultdict(collections.Counter)
    for fold in folds:
        _, g = load(fold)
        for spans in g.values():
            for s, e, l, ent in spans:
                lab_of[ent][l] += 1
    gaz = {ent: next(iter(c)) for ent, c in lab_of.items()
           if len(c) == 1 and sum(c.values()) >= min_count and len(ent) >= min_len}
    return sorted(gaz.items(), key=lambda kv: -len(kv[0]))          # longest strings first

def gazetteer_add(text, spans, gaz, use_labels):
    """Add exact, word-bounded matches of gazetteer strings that do not overlap an existing span."""
    out = [list(x[:3]) for x in spans]; occ = [(s, e) for s, e, *_ in out]
    for ent, lab in gaz:
        if lab not in use_labels:
            continue
        for m in re.finditer(r"(?<!\w)" + re.escape(ent) + r"(?!\w)", text):
            s, e = m.start(), m.end()
            if not any(a < e and b > s for a, b in occ):
                out.append([s, e, lab]); occ.append((s, e))
    return sorted(out)

gaz_f1 = build_gazetteer(["fold1"])
print("gazetteer entries from fold1:", len(gaz_f1), collections.Counter(l for _, l in gaz_f1).most_common())
base_f, base_per = strict_macro_f1(dev_g, pred_r)
print(f"\nbase on dev fold2 (submission-1 fold-1 model + role rules): {base_f:.4f}")
for lab in ("AUTH", "COURT", "STAT", "PREC", "CASENO", "DATE", "JUDGE"):
    p2 = {d: gazetteer_add(dev_t[d], p, gaz_f1, {lab}) for d, p in pred_r.items()}
    f, per = strict_macro_f1(dev_g, p2)
    print(f"+ gazetteer[{lab:6s}] macro {f:.4f} ({f - base_f:+.4f}) | {lab} {base_per[lab]:.3f} -> {per[lab]:.3f}")

gazetteer entries from fold1: 462 [('DATE', 129), ('PREC', 75), ('STAT', 69), ('AUTH', 34), ('CASENO', 30), ('JUDGE', 29), ('COURT', 27), ('RESP', 25), ('APP', 21), ('R.COUNSEL', 9), ('A.COUNSEL', 7), ('WIT', 7)]

base on dev fold2 (submission-1 fold-1 model + role rules): 0.5053
+ gazetteer[AUTH  ] macro 0.5016 (-0.0037) | AUTH 0.376 -> 0.331
+ gazetteer[COURT ] macro 0.5053 (+0.0000) | COURT 0.808 -> 0.808
+ gazetteer[STAT  ] macro 0.5052 (-0.0001) | STAT 0.793 -> 0.791
+ gazetteer[PREC  ] macro 0.5053 (+0.0000) | PREC 0.417 -> 0.417
+ gazetteer[CASENO] macro 0.5053 (+0.0000) | CASENO 0.526 -> 0.526
+ gazetteer[DATE  ] macro 0.5053 (+0.0000) | DATE 0.937 -> 0.937
+ gazetteer[JUDGE ] macro 0.5052 (-0.0002) | JUDGE 0.557 -> 0.556


In [45]:
# ===== B. two-model ensemble on dev: submission-1 style (plain windows) + header windows =====
@torch.no_grad()
def predict_logits_plain(model, ids, bidir, B=8):            # the ORIGINAL window stitching
    wins = make_windows(ids)
    n = len(ids); out = torch.zeros(n, len(TAGS)); best = torch.full((n,), float("inf"))
    model.eval()
    for i in range(0, len(wins), B):
        chunk = wins[i:i + B]
        x, _, pl = collate([(None, s, w, None) for s, w, _ in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(x, pl if bidir else None).float().cpu()
        for j, (s, w, _) in enumerate(chunk):
            L = len(w); pos = torch.arange(L); dist = (pos - (L - 1) / 2).abs()
            idx = s + pos; better = dist < best[idx]
            out[idx[better]] = lg[j, :L][better]; best[idx[better]] = dist[better]
    return out

def ens_predict(members, texts, known):
    """members: list of (model, logits_fn). Averages per-token log-probs, then the same Viterbi decoding."""
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = sum(torch.log_softmax(fn(m, ids, True), -1) for m, fn in members) / len(members)
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp.numpy(), cont), known))
    return pred

model_hdr1 = model                                            # from EXPERIMENT 3
enc_o, ecfg_o = load_encoder(CK_A3)
model_old1 = Tagger(enc_o, ecfg_o.d_model, len(TAGS), dropout=0.1).to(device)
model_old1.load_state_dict(torch.load(glob.glob("/kaggle/**/tagger_arch3mlm_fold1.pt", recursive=True)[0], map_location="cpu"))

k1 = build_known(["fold1"])
pred_e = ens_predict([(model_old1, predict_logits_plain), (model_hdr1, predict_logits)], dev_t, k1)
pred_er = {d: apply_roles(dev_t[d], p) for d, p in pred_e.items()}
fe, pere = strict_macro_f1(dev_g, pred_er)
print(f"DEV fold2 + roles | sub-1 model {base_f:.4f} | header model {fr:.4f} | ENSEMBLE {fe:.4f}")
for l in labels: print(f"  {l:10s} old {base_per[l]:.3f} | hdr {perr[l]:.3f} | ens {pere[l]:.3f}")

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
DEV fold2 + roles | sub-1 model 0.5053 | header model 0.4999 | ENSEMBLE 0.5063
  APP        old 0.272 | hdr 0.241 | ens 0.261
  RESP       old 0.186 | hdr 0.189 | ens 0.180
  A.COUNSEL  old 0.417 | hdr 0.389 | ens 0.396
  R.COUNSEL  old 0.455 | hdr 0.446 | ens 0.469
  JUDGE      old 0.557 | hdr 0.601 | ens 0.589
  WIT        old 0.321 | hdr 0.265 | ens 0.286
  AUTH       old 0.376 | hdr 0.373 | ens 0.382
  COURT      old 0.808 | hdr 0.819 | ens 0.817
  STAT       old 0.793 | hdr 0.781 | ens 0.785
  PREC       old 0.417 | hdr 0.419 | ens 0.423
  DATE       old 0.937 | hdr 0.943 | ens 0.943
  CASENO     old 0.526 | hdr 0.530 | ens 0.544


In [46]:

# ===== C. submission 6: ensemble of the two BOTH-folds models (+ chosen gazetteer labels) =====
GAZ_LABELS = set()          # <- the labels that helped in step A, e.g. {"AUTH"}; leave empty if none did
del model_old1, model_hdr1; torch.cuda.empty_cache()
def load_tagger(path):
    e, c = load_encoder(CK_A3); m = Tagger(e, c.d_model, len(TAGS), dropout=0.1).to(device)
    m.load_state_dict(torch.load(path, map_location="cpu")); return m.eval()
m_old = load_tagger(glob.glob("/kaggle/**/tagger_arch3mlm_both.pt", recursive=True)[0])
m_hdr = load_tagger("/kaggle/working/tagger_arch3mlm_hdr_both.pt")
t3 = json.load(open(P["Test/fold3-text.json"]))
k12 = build_known(["fold1", "fold2"]); gaz12 = build_gazetteer(["fold1", "fold2"])
pred6 = ens_predict([(m_old, predict_logits_plain), (m_hdr, predict_logits)], t3, k12)
pred6 = {d: apply_roles(t3[d], p) for d, p in pred6.items()}
if GAZ_LABELS:
    pred6 = {d: gazetteer_add(t3[d], p, gaz12, GAZ_LABELS) for d, p in pred6.items()}
json.dump(pred6, open("/kaggle/working/prediction.json", "w"))
print("SUBMISSION 6 | docs:", len(pred6), "| entities:", sum(len(v) for v in pred6.values()), "| empty doc:", pred6.get("1305957"))

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
SUBMISSION 6 | docs: 35 | entities: 4622 | empty doc: []


In [47]:
json.dump(pred6, open("/kaggle/working/prediction_sub6.json", "w"))
import zipfile
with zipfile.ZipFile("/kaggle/working/submission_sub6.zip", "w") as z:
    z.write("/kaggle/working/prediction_sub6.json", arcname="prediction.json")   # name inside the zip must stay prediction.json
print("SUBMISSION 6 | docs:", len(pred6), "| entities:", sum(len(v) for v in pred6.values()), "| empty doc:", pred6.get("1305957"))


SUBMISSION 6 | docs: 35 | entities: 4622 | empty doc: []


In [48]:
# ===== member 3 (both folds) + 3-model ensemble on test -> submission 7 =====
import zipfile
for _v in ("m_old", "m_hdr", "model_old1", "model_hdr1", "model"):
    if _v in globals(): del globals()[_v]
torch.cuda.empty_cache()

M3_CKPT, M3_BIDIR, M3_SEED = CK_A3, True, 1       # <- YOUR choice: (CK_A2, False, 0) for arch2 as in EXPERIMENT 1,
                                                  #    or (CK_A3, True, 1) for an arch3-MLM seed variant
enc3, ecfg3 = load_encoder(M3_CKPT)
m3 = Tagger(enc3, ecfg3.d_model, len(TAGS), dropout=0.1).to(device)
train_tagger(m3, w1 + w2, "fold2", bidir=M3_BIDIR, known=build_known(["fold1", "fold2"]),
             EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1, MICRO_B=4,
             CLASS_W_POWER=0.2, WD=0.01, seed=M3_SEED)          # plain windows, your usual settings
torch.save(m3.state_dict(), "/kaggle/working/tagger_member3_both.pt")
m3.eval()

def ens_predict_b(members, texts, known):
    """members: list of (model, logits_fn, bidir). Averages per-token log-probs, then the same Viterbi decoding."""
    pred = {}
    for d, text in texts.items():
        if not text.strip(): pred[d] = []; continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = sum(torch.log_softmax(fn(m, ids, b), -1) for m, fn, b in members) / len(members)
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        pred[d] = fix_brackets(text, decode_refined(text, offs, viterbi(logp.numpy(), cont), known))
    return pred

m_old = load_tagger(glob.glob("/kaggle/**/tagger_arch3mlm_both.pt", recursive=True)[0])
m_hdr = load_tagger("/kaggle/working/tagger_arch3mlm_hdr_both.pt")
t3 = json.load(open(P["Test/fold3-text.json"]))
pred7 = ens_predict_b([(m_old, predict_logits_plain, True), (m_hdr, predict_logits, True),
                       (m3, predict_logits_plain, M3_BIDIR)], t3, build_known(["fold1", "fold2"]))
pred7 = {d: apply_roles(t3[d], p) for d, p in pred7.items()}
json.dump(pred7, open("/kaggle/working/prediction_sub7.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission_sub7.zip", "w") as z:
    z.write("/kaggle/working/prediction_sub7.json", arcname="prediction.json")
print("SUBMISSION 7 | docs:", len(pred7), "| entities:", sum(len(v) for v in pred7.values()), "| empty doc:", pred7.get("1305957"))

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
epoch 1/10 | train loss 0.7865 | dev strict macro-F1 0.3414 | 152s
epoch 2/10 | train loss 0.1507 | dev strict macro-F1 0.5835 | 151s
epoch 3/10 | train loss 0.0985 | dev strict macro-F1 0.6949 | 151s
epoch 4/10 | train loss 0.0686 | dev strict macro-F1 0.7558 | 151s
epoch 5/10 | train loss 0.0484 | dev strict macro-F1 0.8370 | 151s
epoch 6/10 | train loss 0.0362 | dev strict macro-F1 0.8601 | 151s
epoch 7/10 | train loss 0.0264 | dev strict macro-F1 0.8794 | 151s
epoch 8/10 | train loss 0.0195 | dev strict macro-F1 0.9011 | 151s
epoch 9/10 | train loss 0.0154 | dev strict macro-F1 0.9158 | 151s
epoch 10/10 | train loss 0.0129 | dev strict macro-F1 0.9225 | 151s
BEST dev F1 0.9225 {'APP': 0.928, 'RESP': 0.884, 'A.COUNSEL': 0.934, 'R.COUNSEL': 0.95, 'JUDGE': 0.87, 'WIT': 0.992, 'AUTH': 0.955, 'COURT': 0.938, 'STAT': 0.914, 'PREC': 0.859, 'DATE': 0.984, 'CASENO': 0.861}
arch3_mlm_best.pt | step 925 | best_va

In [49]:
# ===== soft document-level role voting: dev check with the two fold-1 models =====
for _v in ("m_old", "m_hdr", "m3"):
    if _v in globals(): del globals()[_v]
torch.cuda.empty_cache()

PERSON6 = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT"]
ROLE_COLS = {r: [T2I["B-" + r], T2I["I-" + r]] for r in PERSON6}

def predict_with_logp(members, texts, known):
    """Ensemble prediction that also keeps the averaged per-token log-probs for each document."""
    out = {}
    for d, text in texts.items():
        if not text.strip(): out[d] = ([], None); continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        logp = sum(torch.log_softmax(fn(m, ids, b), -1) for m, fn, b in members) / len(members)
        cont = [False] + [offs[i-1][1] == offs[i][0] and text[offs[i][0]-1].isalnum() and text[offs[i][0]].isalnum()
                          for i in range(1, len(offs))]
        spans = fix_brackets(text, decode_refined(text, offs, viterbi(logp.numpy(), cont), known))
        out[d] = (spans, logp)
    return out

def soft_role_vote(text, offs, logp, spans):
    """For each distinct person-name string: sum, over all its mentions, the mean log-prob of each role
    (B-role + I-role probability per token); assign the winning role to every mention of that name."""
    probs = logp.exp()
    role_p = torch.stack([probs[:, ROLE_COLS[r]].sum(-1) for r in PERSON6], -1)       # (n_tokens, 6)
    starts = [s for s, _ in offs]; ends = [e for _, e in offs]
    votes = {}
    for s, e, l in spans:
        if l not in PERSON6: continue
        i0 = bisect.bisect_right(ends, s); i1 = bisect.bisect_left(starts, e)
        if i1 <= i0: continue
        v = role_p[i0:i1].clamp_min(1e-6).log().mean(0)
        votes[text[s:e]] = votes.get(text[s:e], torch.zeros(len(PERSON6))) + v
    return [[s, e, PERSON6[int(votes[text[s:e]].argmax())] if (l in PERSON6 and text[s:e] in votes) else l]
            for s, e, l in spans]

m_o1 = load_tagger(glob.glob("/kaggle/**/tagger_arch3mlm_fold1.pt", recursive=True)[0])
m_h1 = load_tagger("/kaggle/working/tagger_arch3mlm_hdr_fold1.pt")
R = predict_with_logp([(m_o1, predict_logits_plain, True), (m_h1, predict_logits, True)], dev_t, build_known(["fold1"]))

v_rules = {d: apply_roles(dev_t[d], sp) for d, (sp, _) in R.items()}                       # current pipeline
v_soft = {d: (soft_role_vote(dev_t[d], ENC[d][1], lp, sp) if lp is not None else sp) for d, (sp, lp) in R.items()}
v_soft_rules = {d: apply_roles(dev_t[d], v_soft[d]) for d in v_soft}                          # soft vote, then cue rules
print(f"{'variant':22s} {'macro':>7s} " + " ".join(f"{r:>9s}" for r in PERSON6))
for name, pr in (("hard rules (current)", v_rules), ("soft vote", v_soft), ("soft vote + rules", v_soft_rules)):
    f, per = strict_macro_f1(dev_g, pr)
    print(f"{name:22s} {f:7.4f} " + " ".join(f"{per[r]:9.3f}" for r in PERSON6))

arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
variant                  macro       APP      RESP A.COUNSEL R.COUNSEL     JUDGE       WIT
hard rules (current)    0.5063     0.261     0.180     0.396     0.469     0.589     0.286
soft vote               0.4854     0.208     0.167     0.296     0.397     0.587     0.276
soft vote + rules       0.5047     0.252     0.178     0.418     0.452     0.587     0.276


In [50]:
# ===== merge fragmented long entities (PREC / CASENO / STAT ...): dev check on the current pipeline =====
GAP_OK = re.compile(r"^[\s,;:]*(?:v\.?|vs\.?|versus|and|&|of|in|nos?\.?)?[\s,;:]*$", re.I)

def merge_spans(text, spans, labs, max_gap):
    """Join consecutive same-label spans (label in labs) separated by <= max_gap chars of only
    whitespace/punctuation or one connector word (v., vs., versus, and, &, of, in, No./Nos.)."""
    out = []
    for s, e, l in sorted([list(x[:3]) for x in spans]):
        if (out and l in labs and out[-1][2] == l and 0 <= s - out[-1][1] <= max_gap
                and GAP_OK.match(text[out[-1][1]:s])):
            out[-1][1] = e
        else:
            out.append([s, e, l])
    return out

base_f, base_p = strict_macro_f1(dev_g, v_rules)          # current best dev pipeline (ensemble + hard rules)
print(f"base {base_f:.4f} | PREC {base_p['PREC']:.3f} CASENO {base_p['CASENO']:.3f} STAT {base_p['STAT']:.3f} AUTH {base_p['AUTH']:.3f} COURT {base_p['COURT']:.3f}")
for labs in ({"PREC"}, {"CASENO"}, {"STAT"}, {"AUTH"}, {"COURT"}):
    for mg in (2, 6, 12):
        pr = {d: merge_spans(dev_t[d], p, labs, mg) for d, p in v_rules.items()}
        f, per = strict_macro_f1(dev_g, pr)
        lab = next(iter(labs))
        print(f"merge {lab:6s} gap<={mg:2d}: macro {f:.4f} ({f - base_f:+.4f}) | {lab} {base_p[lab]:.3f} -> {per[lab]:.3f}")

base 0.5063 | PREC 0.423 CASENO 0.544 STAT 0.785 AUTH 0.382 COURT 0.817
merge PREC   gap<= 2: macro 0.5047 (-0.0016) | PREC 0.423 -> 0.403
merge PREC   gap<= 6: macro 0.5036 (-0.0027) | PREC 0.423 -> 0.390
merge PREC   gap<=12: macro 0.5036 (-0.0027) | PREC 0.423 -> 0.390
merge CASENO gap<= 2: macro 0.5028 (-0.0035) | CASENO 0.544 -> 0.502
merge CASENO gap<= 6: macro 0.4971 (-0.0093) | CASENO 0.544 -> 0.433
merge CASENO gap<=12: macro 0.4971 (-0.0093) | CASENO 0.544 -> 0.433
merge STAT   gap<= 2: macro 0.5066 (+0.0003) | STAT 0.785 -> 0.788
merge STAT   gap<= 6: macro 0.5066 (+0.0002) | STAT 0.785 -> 0.788
merge STAT   gap<=12: macro 0.5066 (+0.0002) | STAT 0.785 -> 0.788
merge AUTH   gap<= 2: macro 0.5059 (-0.0005) | AUTH 0.382 -> 0.377
merge AUTH   gap<= 6: macro 0.5059 (-0.0005) | AUTH 0.382 -> 0.377
merge AUTH   gap<=12: macro 0.5059 (-0.0005) | AUTH 0.382 -> 0.377
merge COURT  gap<= 2: macro 0.5063 (-0.0000) | COURT 0.817 -> 0.817
merge COURT  gap<= 6: macro 0.5064 (+0.0000) | COU

In [51]:
# ===== submission 9: knowledge distillation — ensemble teacher (sub-7 labels on test text) -> student =====
import zipfile
for _v in ("m_o1", "m_h1", "m_old", "m_hdr", "m3", "model"):
    if _v in globals(): del globals()[_v]
torch.cuda.empty_cache()

t3 = json.load(open(P["Test/fold3-text.json"]))

def build_pseudo_split(texts, teacher_preds):
    """Plain windows over the test documents, tagged with the teacher's (sub-7 ensemble) spans."""
    data = []
    for d, text in texts.items():
        if not text.strip(): continue
        if d not in ENC: ENC[d] = encode_with_offsets(text)
        ids, offs = ENC[d]
        tags = tag_doc(text, offs, [[s, e, l] for s, e, l, *_ in teacher_preds[d]])
        for s, wi, wt in make_windows(ids, tags):
            data.append((d, s, wi, wt))
    return data

wp = build_pseudo_split(t3, pred7)
print(f"pseudo-labelled test windows: {len(wp)} | gold windows: {len(w1) + len(w2)} | "
      f"teacher entities: {sum(len(v) for v in pred7.values())}")

enc_s, ecfg_s = load_encoder(CK_A3)
student = Tagger(enc_s, ecfg_s.d_model, len(TAGS), dropout=0.1).to(device)
train_tagger(student, w1 + w2 + wp, "fold2", bidir=True, known=build_known(["fold1", "fold2"]),
             EPOCHS=10, LR_ENC=1e-5, LR_HEAD=1e-4, WARMUP_FRAC=0.1, MICRO_B=4,
             CLASS_W_POWER=0.2, WD=0.01, seed=2)        # your usual settings; "dev" F1 printed is on training data
torch.save(student.state_dict(), "/kaggle/working/tagger_student_kd.pt")
student.eval()

m_old = load_tagger(glob.glob("/kaggle/**/tagger_arch3mlm_both.pt", recursive=True)[0])
m_hdr = load_tagger("/kaggle/working/tagger_arch3mlm_hdr_both.pt")
m3 = load_tagger("/kaggle/working/tagger_member3_both.pt")
members = [(m_old, predict_logits_plain, True), (m_hdr, predict_logits, True),
           (m3, predict_logits_plain, True), (student, predict_logits_plain, True)]
pred9 = ens_predict_b(members, t3, build_known(["fold1", "fold2"]))
pred9 = {d: apply_roles(t3[d], p) for d, p in pred9.items()}
json.dump(pred9, open("/kaggle/working/prediction_sub9.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission_sub9.zip", "w") as z:
    z.write("/kaggle/working/prediction_sub9.json", arcname="prediction.json")
print("SUBMISSION 9 | docs:", len(pred9), "| entities:", sum(len(v) for v in pred9.values()), "| empty doc:", pred9.get("1305957"))

# student alone too (a clean KD data point for the report)
pred9s = {d: apply_roles(t3[d], p) for d, p in ens_predict_b([(student, predict_logits_plain, True)], t3,
                                                            build_known(["fold1", "fold2"])).items()}
json.dump(pred9s, open("/kaggle/working/prediction_student_only.json", "w"))
print("student-only predictions saved (optional extra submission)")

pseudo-labelled test windows: 455 | gold windows: 848 | teacher entities: 4633
arch3_mlm_best.pt | step 925 | best_val 2.043 | vocab 65356 | 184,910,848 params
epoch 1/10 | train loss 0.6433 | dev strict macro-F1 0.4321 | 220s
epoch 2/10 | train loss 0.1174 | dev strict macro-F1 0.5975 | 220s
epoch 3/10 | train loss 0.0723 | dev strict macro-F1 0.7258 | 220s
epoch 4/10 | train loss 0.0502 | dev strict macro-F1 0.7877 | 220s
epoch 5/10 | train loss 0.0354 | dev strict macro-F1 0.8301 | 220s
epoch 6/10 | train loss 0.0246 | dev strict macro-F1 0.8795 | 220s
epoch 7/10 | train loss 0.0185 | dev strict macro-F1 0.8916 | 220s
epoch 8/10 | train loss 0.0134 | dev strict macro-F1 0.9149 | 220s
epoch 9/10 | train loss 0.0104 | dev strict macro-F1 0.9300 | 220s
epoch 10/10 | train loss 0.0083 | dev strict macro-F1 0.9345 | 220s
BEST dev F1 0.9345 {'APP': 0.939, 'RESP': 0.882, 'A.COUNSEL': 0.953, 'R.COUNSEL': 0.965, 'JUDGE': 0.883, 'WIT': 0.988, 'AUTH': 0.966, 'COURT': 0.949, 'STAT': 0.934, 'PRE

In [52]:
# ===== confidence filter: drop low-probability spans (precision lever) — dev check =====
def span_conf(starts, ends, logp, s, e, l):
    i0 = bisect.bisect_right(ends, s); i1 = bisect.bisect_left(starts, e)
    if i1 <= i0: return 1.0
    p = logp[i0:i1].exp()
    return float(p[:, [T2I["B-" + l], T2I["I-" + l]]].sum(-1).mean())

def conf_filter(offs, logp, spans, tau):
    starts = [a for a, _ in offs]; ends = [b for _, b in offs]
    return [[s, e, l] for s, e, l in spans if span_conf(starts, ends, logp, s, e, l) >= tau]

def macro_pr(gold, pred):
    tp, fp, fn = collections.Counter(), collections.Counter(), collections.Counter()
    for d in gold:
        G = {(s, e, l) for s, e, l, *_ in gold[d]}; Pd = {(s, e, l) for s, e, l, *_ in pred.get(d, [])}
        for x in Pd: (tp if x in G else fp)[x[2]] += 1
        for x in G - Pd: fn[x[2]] += 1
    P = sum(tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0 for l in labels) / len(labels)
    Rr = sum(tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0 for l in labels) / len(labels)
    return P, Rr

base = {d: apply_roles(dev_t[d], sp) for d, (sp, _) in R.items()}
bf, _ = strict_macro_f1(dev_g, base); bP, bR = macro_pr(dev_g, base)
print(f"base             macro-F1 {bf:.4f} | macro-P {bP:.3f} | macro-R {bR:.3f}")
for tau in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    pr = {d: (apply_roles(dev_t[d], conf_filter(ENC[d][1], lp, sp, tau)) if lp is not None else sp)
          for d, (sp, lp) in R.items()}
    f, _ = strict_macro_f1(dev_g, pr); Pp, Rp = macro_pr(dev_g, pr)
    print(f"conf >= {tau:.1f}     macro-F1 {f:.4f} ({f - bf:+.4f}) | macro-P {Pp:.3f} | macro-R {Rp:.3f}")

base             macro-F1 0.5063 | macro-P 0.496 | macro-R 0.533
conf >= 0.3     macro-F1 0.5060 (-0.0003) | macro-P 0.496 | macro-R 0.532
conf >= 0.4     macro-F1 0.5053 (-0.0010) | macro-P 0.499 | macro-R 0.528
conf >= 0.5     macro-F1 0.5072 (+0.0009) | macro-P 0.516 | macro-R 0.517
conf >= 0.6     macro-F1 0.4995 (-0.0068) | macro-P 0.537 | macro-R 0.494
conf >= 0.7     macro-F1 0.4784 (-0.0279) | macro-P 0.552 | macro-R 0.459
conf >= 0.8     macro-F1 0.4550 (-0.0513) | macro-P 0.590 | macro-R 0.424


In [53]:
# ===== submission 10: sub-9 ensemble + confidence filter =====
import zipfile
TAU = 0.5          # <- YOUR pick from cell 1
assert TAU is not None
members = [(m_old, predict_logits_plain, True), (m_hdr, predict_logits, True),
           (m3, predict_logits_plain, True), (student, predict_logits_plain, True)]
RT = predict_with_logp(members, t3, build_known(["fold1", "fold2"]))
pred10 = {d: (apply_roles(t3[d], conf_filter(ENC[d][1], lp, sp, TAU)) if lp is not None else sp)
          for d, (sp, lp) in RT.items()}
json.dump(pred10, open("/kaggle/working/prediction_sub10.json", "w"))
with zipfile.ZipFile("/kaggle/working/submission_sub10.zip", "w") as z:
    z.write("/kaggle/working/prediction_sub10.json", arcname="prediction.json")
print("SUBMISSION 10 | docs:", len(pred10), "| entities:", sum(len(v) for v in pred10.values()),
      f"(sub9: {sum(len(v) for v in pred9.values())}) | empty doc:", pred10.get("1305957"))

SUBMISSION 10 | docs: 35 | entities: 4386 (sub9: 4654) | empty doc: []


In [ ]:
# ===== PEFT plumbing: freeze the backbone, report what is trainable =====
def freeze_all_but(model, keep):
    """keep(name, param) -> True if this parameter should train."""
    for n, p in model.named_parameters():
        p.requires_grad = bool(keep(n, p))

def trainable_report(model, tag=""):
    seen = {id(p): p for p in model.parameters()}.values()
    tot = sum(p.numel() for p in seen); tr = sum(p.numel() for p in seen if p.requires_grad)
    print(f"{tag} trainable {tr:,} / {tot:,} params ({tr / tot:.3%})")

In [ ]:
import torch
import torch.nn as nn
import math

class LoRMALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int, alpha: float):
        super().__init__()
        self.base = base
        
        for param in self.base.parameters():
            param.requires_grad = False
            
        d_out = base.out_features
        
        self.lorma_A = nn.Parameter(torch.empty(r, d_out))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        
        self.lorma_B = nn.Parameter(torch.zeros(d_out, r))
        
        self.scale = alpha / r

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * (h @ self.lorma_A.T @ self.lorma_B.T)

In [ ]:
# ===== arch 2 + LoRMA: inject, check, train, submit =====
import zipfile

def inject(model, wrap_fn, targets):
    """Replace every nn.Linear child whose attribute name is in `targets` with wrap_fn(linear)."""
    n = 0
    for _, mod in list(model.named_modules()):
        for t in targets:
            child = getattr(mod, t, None)
            if isinstance(child, nn.Linear):
                setattr(mod, t, wrap_fn(child)); n += 1
    return n

# ---- YOUR choices (state them in the report) ----
LORMA_R, LORMA_ALPHA = None, None
LORMA_TARGETS = ("q_proj", "k_proj", "v_proj", "out_proj")
LR_PEFT = None
TRAIN_FOLDS = ["fold1", "fold2"]
assert None not in (LORMA_R, LORMA_ALPHA, LR_PEFT)

for _v in ("m_old", "m_hdr", "m3", "student", "m_o1", "m_h1", "model", "m_a2"):
    if _v in globals(): del globals()[_v]
torch.cuda.empty_cache()

enc2, ecfg2 = load_encoder(CK_A2)
m_lorma = Tagger(enc2, ecfg2.d_model, len(TAGS), dropout=0.1)
x_chk = torch.tensor([w1[0][2][:64]])
with torch.no_grad():
    m_lorma.eval(); ref = m_lorma(x_chk)
n_wrapped = inject(m_lorma, lambda lin: LoRMALinear(lin, LORMA_R, LORMA_ALPHA), LORMA_TARGETS)
with torch.no_grad():
    same = torch.allclose(ref, m_lorma(x_chk), atol=1e-5)
print(f"wrapped {n_wrapped} Linears | output unchanged at init (B=0): {same}")
assert same, "LoRMA must start as the identity -- check B's zero init and the forward pass"

freeze_all_but(m_lorma, lambda n, p: ("lorma" in n) or n.startswith("cls."))
trainable_report(m_lorma, "arch2+LoRMA")
m_lorma = m_lorma.to(device)

predict_logits_hdr = predict_logits; predict_logits = predict_logits_plain
w_train = [w for f in TRAIN_FOLDS for w in build_split([f])]
f_dev = train_tagger(m_lorma, w_train, "fold2", bidir=False, known=build_known(TRAIN_FOLDS),
                     EPOCHS=10, LR_ENC=LR_PEFT, LR_HEAD=1e-4, WARMUP_FRAC=0.1, MICRO_B=4,
                     CLASS_W_POWER=0.2, WD=0.01)
predict_logits = predict_logits_hdr
tagname = "arch2_lorma_" + "_".join(TRAIN_FOLDS)
torch.save({k: v for k, v in m_lorma.state_dict().items() if "lorma" in k or k.startswith("cls.")},
           f"/kaggle/working/{tagname}_adapters.pt")

if TRAIN_FOLDS == ["fold1"]:
    pv = ens_predict_b([(m_lorma, predict_logits_plain, False)], dev_t, build_known(["fold1"]))
    fd, perd = strict_macro_f1(dev_g, {d: apply_roles(dev_t[d], p) for d, p in pv.items()})
    print(f"DEV fold2 (+roles): {fd:.4f}")
else:
    t3 = json.load(open(P["Test/fold3-text.json"]))
    pt = ens_predict_b([(m_lorma, predict_logits_plain, False)], t3, build_known(TRAIN_FOLDS))
    pt = {d: apply_roles(t3[d], p) for d, p in pt.items()}
    json.dump(pt, open(f"/kaggle/working/prediction_{tagname}.json", "w"))
    with zipfile.ZipFile(f"/kaggle/working/submission_{tagname}.zip", "w") as z:
        z.write(f"/kaggle/working/prediction_{tagname}.json", arcname="prediction.json")
    print("ARCH2+LoRMA | docs:", len(pt), "| entities:", sum(len(v) for v in pt.values()), "| empty doc:", pt.get("1305957"))